<a href="https://colab.research.google.com/github/cruhling289/MSDSCapstone/blob/main/notebooks/MSDSCapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**importing the llm (qwen3) through the transformers library**

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", device_map="auto")
messages = [
    {"role": "user", "content": prompt},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	enable_thinking=False,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=300)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

KeyboardInterrupt: 

**Creating text variable with the content**

In [1]:
!wget -q https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md
with open("capstone_day_planning.md", "r") as f:
    text = f.read()

#print(text[:2000])

**another way to read the text**

In [2]:
import requests
try:
  rawURL = "https://raw.githubusercontent.com/cruhling289/MSDSCapstone/main/data/capstone_day_planning.md"
  response = requests.get(rawURL, timeout=10)
  response.raise_for_status
  text = response.text
except Exception as e:
  print(f'Unexpected error: {e}')

#print(text)

**Manually chunking text, fix later to make it automatic**

In [3]:
chunks = [
    "## Event and Venue Info",
    "## Preparing for the Event",
    "## Attendance",
    "## Awards",
    "## Presentation Schedule",
]
chunkList = []
currentChunk = ""

for line in text.splitlines():

    if line in chunks:
        if currentChunk != "":
          chunkList.append(currentChunk)
        currentChunk = line + "\n"
    else:
        currentChunk += line + "\n"
chunkList.append(currentChunk)




In [4]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

**Embedding chunked text**

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
#embedding_practice = embedding_model.encode("Which talks happen before 11AM")
#print(embedding_practice)
#print(len(embedding_practice))
embeddings = []
for chunk in chunkList:
  x = embedding_model.encode(chunk)
  embeddings.append(x)
#print(len(embeddings))
#print(len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

**Vector similarity/retrieval**

In [7]:
query = "Which talks happen before 11am?"
queryEmbedding = embedding_model.encode(query)
similarities = []
for embedding in embeddings:
  similarity = cosine_similarity([queryEmbedding], [embedding]) [0][0]
  similarities.append(similarity)
#print((similarities))

top_indices = sorted(
    range(len(similarities)),
    key=lambda i: similarities[i],
    reverse=True
)[:2]

retrievedChunks = []
for i in top_indices:
  retrievedChunks.append(chunkList[i])

**Combining the k retrieved chunks**

In [8]:
joinedRetrieval = ""
for i in range(len(retrievedChunks)):
  joinedRetrieval += retrievedChunks[i] + "\n\n"


**Generation**

In [9]:
prompt = f'''Using only the following context, answer the question.
  Context: {joinedRetrieval} \n
  Question: {query}\n
  Answer: '''
#print(prompt)